In [1]:
print("Hola Fastapi")

Hola Fastapi


In [2]:
#importaciones

import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
    

print(RAIZ)
print("CSV ventas:", (RAIZ / "data" / "raw" / "ventas.csv").exists())
print("Carpeta modelos:", RAIZ / "modelos")

c:\Programacion\DataScienceIA\Docente_Dev_Senior_code\Clases_Master_IA\Modulo_2\Unidad_1\Clase_2_persistencia
CSV ventas: True
Carpeta modelos: c:\Programacion\DataScienceIA\Docente_Dev_Senior_code\Clases_Master_IA\Modulo_2\Unidad_1\Clase_2_persistencia\modelos


In [4]:
import runpy

from src.features_modelo import COLUMNAS_FEATURES, COLUMNA_TARGET

print("Columnas que entraran al modelo (X): ", COLUMNAS_FEATURES )
print("target (y): ", COLUMNA_TARGET)

runpy.run_path(str(RAIZ / "scripts" / "entrenar_modelos.py"), run_name="__main__")

carpeta = RAIZ / "modelos"

esperado = [
    "gasto_sklearn.joblib",
    "gasto_mlp.joblib",
    "features_clientes.csv",
    "meta.json"
]

print("\n Inventario de modelos:")
for nombre in esperado:
    ruta = carpeta / nombre
    print(f"f OK  {nombre:28} {ruta.stat().st_size / 1024:8.1f} KB")



Columnas que entraran al modelo (X):  ['num_compras', 'gasto_historico', 'ticket_medio', 'cantidad_total']
target (y):  gasto_siguiente_periodo
Construyendo features desde data/raw/*.csv ...
  filas=9,996  columnas=['num_compras', 'gasto_historico', 'ticket_medio', 'cantidad_total']

1) RandomForestRegressor (scikit-learn clasico)
   MAE=1016.89  R2=0.049  -> gasto_sklearn.joblib

2) MLPRegressor (red neuronal de scikit-learn)
   MAE=1003.38  R2=0.062  -> gasto_mlp.joblib

Lookup clientes -> features_clientes.csv (9,996 filas)
Metadatos -> meta.json

Listo. La API cargara estos archivos al arrancar.

 Inventario de modelos:
f OK  gasto_sklearn.joblib            853.1 KB
f OK  gasto_mlp.joblib                 17.6 KB
f OK  features_clientes.csv           321.2 KB
f OK  meta.json                         0.4 KB


In [5]:
import pandas as pd

lookup = pd.read_csv(RAIZ / "modelos" / "features_clientes.csv")

print("Filas en el lookup ( 1 por cliente):", len(lookup))

print(lookup.head(3).to_string(index=False))
print("\n Entrenamiento cerrado. El resto de la clase solo cargamos y servimos")

Filas en el lookup ( 1 por cliente): 9996
 cliente_id  num_compras  gasto_historico  ticket_medio  cantidad_total  gasto_siguiente_periodo
          1           13          5804.36        446.49              37                    59.23
          2            9          2484.18        276.02              14                   715.29
          3           11          1809.19        164.47              24                    48.19

 Entrenamiento cerrado. El resto de la clase solo cargamos y servimos


In [6]:
# Cargar un joblib a mano (lo mismo que haría FastAPI al arrancar).
# lifespan() en api/main.py llama a cargar_paquete() UNA vez.
# No se reentrena ni se hace joblib.load en cada petición.

import joblib
import pandas as pd

from api.carga_modelos import cargar_lookup, cargar_paquete
from src.features_modelo import COLUMNAS_FEATURES

ruta = RAIZ / "modelos" / "gasto_sklearn.joblib"
paquete = joblib.load(ruta)

print("Archivo :", ruta.name)
print("Claves  :", list(paquete.keys()))
print("Tipo    :", type(paquete["modelo"]).__name__)
print("Features:", paquete["features"])
print("Version :", paquete["version"])
print("Metricas:", paquete["metricas"])

# La API hace exactamente esto en el lifespan (y además carga el MLP y el lookup).
sklearn = cargar_paquete("gasto_sklearn.joblib")
mlp = cargar_paquete("gasto_mlp.joblib")
lookup = cargar_lookup()

print("\nEn memoria, como ESTADO al arrancar uvicorn:")
print("  sklearn :", type(sklearn["modelo"]).__name__, "v" + sklearn["version"])
print("  mlp     :", type(mlp["modelo"]).__name__, "v" + mlp["version"])
print("  lookup  :", len(lookup), "clientes")

# Inferencia de 1 fila, mismo orden de columnas que en el entrenamiento.
cliente_id = 1
nums = lookup.loc[cliente_id]
x = pd.DataFrame(
    [{
        "num_compras": int(nums["num_compras"]),
        "gasto_historico": float(nums["gasto_historico"]),
        "ticket_medio": float(nums["ticket_medio"]),
        "cantidad_total": int(nums["cantidad_total"]),
    }],
    columns=COLUMNAS_FEATURES,
)
valor = float(sklearn["modelo"].predict(x)[0])
print(f"\nCliente {cliente_id} -> gasto predicho (sklearn): {valor:.2f}")


Archivo : gasto_sklearn.joblib
Claves  : ['modelo', 'features', 'target', 'version', 'metricas']
Tipo    : RandomForestRegressor
Features: ['num_compras', 'gasto_historico', 'ticket_medio', 'cantidad_total']
Version : 1.0.0
Metricas: {'mae': 1016.89, 'r2': 0.049, 'segundos': 0.38}

En memoria, como ESTADO al arrancar uvicorn:
  sklearn : RandomForestRegressor v1.0.0
  mlp     : Pipeline v1.0.0
  lookup  : 9996 clientes

Cliente 1 -> gasto predicho (sklearn): 698.84


# PyDantic : El portero

In [ ]:
from pydantic import ValidationError

from api.esquemas import EntradaPrediccion

ok = EntradaPrediccion.model_validate(
    {
        "cliente_id": 42,
        "num_compras": 18,
        "ticket_medio": 116.7,
        "cantidad_total": 40,
    }
)

print("OK - Objetivo validado")
print(ok)
print("Dict / Json que viajaría:", ok.model_dump())

